In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch, RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables.graph_ascii import draw_ascii
from typing import Literal
from pydantic import BaseModel
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
os.environ['OLLAMA_API_KEY'] = apiKey
llm = ChatOllama(model="gpt-oss:120b-cloud")

In [5]:
# Example 1 - Sequential Chains with pipes
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain = prompt_1 | llm | parser | prompt_2 | llm | parser
response = chain.invoke({"country": "France"})

print(response)

Here are five of the most famous tourist attractions you’ll find in **Paris**, the capital city of France:

| # | Tourist Site | Why It’s Iconic |
|---|--------------|----------------|
| 1 | **Eiffel Tower** | The global symbol of Paris; offers stunning panoramic views from its three observation decks. |
| 2 | **Louvre Museum** | The world’s largest art museum, home to masterpieces such as the *Mona Lisa* and the *Venus de Milo*. |
| 3 | **Cathédrale Notre‑Dame de Paris** | A masterpiece of French Gothic architecture (currently undergoing restoration after the 2019 fire). |
| 4 | **Champs‑Élysées & Arc de Triomphe** | The grand avenue lined with shops and cafés, culminating in the monumental arch that honors France’s military victories. |
| 5 | **Montmartre & Sacré‑Coeur Basilica** | A historic hilltop neighborhood known for its artistic heritage; the white-domed basilica offers sweeping city vistas. |

These sites capture the art, history, and romance that make Paris a top destination

In [6]:
# Example 2 - Sequential Chains with RunnableSequence
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain_1 = prompt_1 | llm | parser
chain_2 = prompt_2 | llm | parser
sequence = RunnableSequence(chain_1, chain_2)
response = sequence.invoke({"country": "China"})

print(response)

Here are five of the most iconic tourist attractions in **Beijing**, each offering a glimpse into the city’s rich history, culture, and modern vibrancy:

| # | Tourist Place | What Makes It Famous |
|---|----------------|----------------------|
| 1 | **The Great Wall (Mutianyu or Badaling sections)** | One of the Seven Wonders of the World, the Wall winds over mountains and valleys about 70 km from downtown Beijing. The Mutianyu section is especially scenic and less crowded, while Badaling is the most visited and easily accessible. |
| 2 | **The Forbidden City (Palace Museum)** | The imperial palace of the Ming and Qing dynasties, covering 720,000 m² with nearly 1,000 buildings. It houses priceless artworks, throne rooms, and the iconic Hall of Supreme Harmony. |
| 3 | **Tiananmen Square** | The world’s largest city square, surrounded by the **Great Hall of the People**, **Mausoleum of Mao Zedong**, and the **National Museum of China**. It’s a focal point for national ceremonies and hi